# Practical worksheet: Autoencoders and Variational Autoencoders (MNIST + Fashion-MNIST)


## Learning focus

1. Train Dense AE on one MNIST digit.
2. Train Dense AE and Convolutional AE on all MNIST digits.
3. Compare metrics and reconstructions across epochs.
4. Build and train a convolutional VAE and compare all 3 models.
5. Perform latent traversal and decoder sampling.
6. Train Conv AE and VAE on Fashion-MNIST.
7. Compare Fashion results with FID (library-based).


## Notebook setup

The next code blocks import dependencies, set reproducibility and hardware (`cuda` / `mps` / `cpu`),
and define reusable data-loading and visualization helpers.

Run all setup blocks once before starting the experiments.


In [ ]:
%pip install -q "torch-fidelity>=0.3,<1.0"
import random
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm


Define reproducibility and hardware selection.

This block fixes randomness and selects `cuda`, `mps`, or `cpu`. Check the printed device before training.


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('Device:', device)


Create reusable data-loading pipeline for MNIST and Fashion-MNIST.

Use `target_class` for one-class experiments, or `None` for all classes.


In [ ]:
def build_digit_loaders(
    dataset_name='mnist',
    target_class=None,
    batch_size=128,
    train_limit=12000,
    test_limit=2000,
):
    transform = transforms.ToTensor()

    if dataset_name.lower() == 'mnist':
        train_ds = datasets.MNIST(root='data', train=True, download=True, transform=transform)
        test_ds = datasets.MNIST(root='data', train=False, download=True, transform=transform)
    elif dataset_name.lower() == 'fashion':
        train_ds = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)
        test_ds = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)
    else:
        raise ValueError("dataset_name must be 'mnist' or 'fashion'")

    if target_class is not None:
        train_idx = torch.where(train_ds.targets == target_class)[0]
        test_idx = torch.where(test_ds.targets == target_class)[0]
    else:
        train_idx = torch.arange(len(train_ds))
        test_idx = torch.arange(len(test_ds))

    if train_limit is not None:
        train_idx = train_idx[:train_limit]
    if test_limit is not None:
        test_idx = test_idx[:test_limit]

    train_subset = Subset(train_ds, train_idx.tolist())
    test_subset = Subset(test_ds, test_idx.tolist())

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


### PyTorch docs for key building blocks used here

- `nn.Conv2d`: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
- `nn.ConvTranspose2d`: https://pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html
- `nn.BatchNorm2d`: https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html
- `nn.Linear`: https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
- `nn.Sequential`: https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html
- `binary_cross_entropy`: https://pytorch.org/docs/stable/generated/torch.nn.functional.binary_cross_entropy.html
- `randn_like`: https://pytorch.org/docs/stable/generated/torch.randn_like.html
- `DataLoader`: https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader
- FID metric (torch-fidelity): https://github.com/toshas/torch-fidelity

Define visualization and reporting helpers used throughout the notebook.

These functions are shared utilities for reconstructions, traversals, and epoch metric plots.


In [ ]:
def plot_batch_examples(loader, title='Samples', n_show=16):
    x, y = next(iter(loader))
    n_show = min(n_show, x.size(0))
    rows = int(np.sqrt(n_show))
    cols = int(np.ceil(n_show / rows))

    fig, axes = plt.subplots(rows, cols, figsize=(1.8 * cols, 1.8 * rows))
    axes = np.array(axes).reshape(rows, cols)
    for i in range(rows * cols):
        ax = axes[i // cols, i % cols]
        if i < n_show:
            ax.imshow(x[i, 0].numpy(), cmap='gray')
            ax.set_title(f'y={int(y[i])}', fontsize=8)
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


MODEL_DIR = Path('artifacts/models')
HISTORY_DIR = Path('artifacts/histories')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

USE_SAVED_MODELS_IF_AVAILABLE = True  # Set to False to force retraining


def _artifact_stem(run_name):
    stem = ''.join(ch.lower() if ch.isalnum() else '_' for ch in run_name)
    stem = '_'.join(part for part in stem.split('_') if part)
    return stem or 'model'


def _artifact_paths(run_name):
    stem = _artifact_stem(run_name)
    return MODEL_DIR / f'{stem}.pt', HISTORY_DIR / f'{stem}.json'


def fit_or_load_model(model, run_name, train_fn, load_if_available=USE_SAVED_MODELS_IF_AVAILABLE, **train_kwargs):
    model_path, history_path = _artifact_paths(run_name)

    if load_if_available and model_path.exists():
        state_dict = torch.load(model_path, map_location=device)
        model.load_state_dict(state_dict)
        model.to(device)
        if history_path.exists():
            with history_path.open('r', encoding='utf-8') as f:
                history = json.load(f)
        else:
            history = []
        print(f'Loaded {run_name} model from {model_path}')
        return history

    history = train_fn(model=model, **train_kwargs)
    with model_path.open('wb') as f:
        torch.save(model.state_dict(), f)
    with history_path.open('w', encoding='utf-8') as f:
        json.dump(history, f, indent=2)
    print(f'Saved {run_name} model to {model_path} and history to {history_path}')
    return history


## 1 - MNIST one-digit only: Dense AE

Instantiate one-digit MNIST loaders.

This controlled setting simplifies the first AE experiment and makes reconstruction behavior easier to inspect.


### Dense AE pseudocode (this section)

```text
input: x (batch of images)
x_flat = flatten(x)
z = Encoder(x_flat)
x_hat = Decoder(z)
L_recon = BCE(x_hat, x_flat)
update parameters with backprop
```

Goal in this first stage: verify the full AE pipeline in a controlled one-digit setting.


In [ ]:
train_one_loader, test_one_loader = build_digit_loaders(
    dataset_name='mnist',
    target_class=1,
    batch_size=128,
    train_limit=8000,
    test_limit=1500,
)
plot_batch_examples(train_one_loader, title='MNIST one-digit')


Define the Dense Autoencoder architecture.

Learners should focus on the encoder bottleneck and mirrored decoder structure.


In [ ]:
class DenseAutoencoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dims=(256, 128), latent_dim=16):
        super().__init__()
        # TODO START
        raise NotImplementedError('Implement DenseAutoencoder layers')
        # TODO END

    def forward(self, x):
        return self.decoder(self.encoder(x))


Define generic train/evaluation loops for deterministic autoencoders.

These functions will be reused for both Dense AE and Convolutional AE.


In [ ]:
def train_autoencoder(model, loader, optimizer, epochs=20, flatten_input=True):
    model.train()
    history = []
    for ep in range(epochs):
        run = 0.0
        for x, _ in tqdm(loader, leave=False):
            x = x.to(device)
            xin = x.view(x.size(0), -1) if flatten_input else x
            # TODO START
            raise NotImplementedError('Implement AE training step')
            # TODO END
            run += loss.item() * x.size(0)
        epoch_recon = run / len(loader.dataset)
        history.append({'train_recon_bce': epoch_recon})
        print(f'Epoch {ep+1}/{epochs} | train_recon_bce={epoch_recon:.4f}')
    return history


def evaluate_autoencoder(model, loader, flatten_input=True):
    model.eval()
    tb, tm, ta, n = 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            xin = x.view(x.size(0), -1) if flatten_input else x
            xhat = model(xin)
            b = x.size(0)
            # TODO START
            raise NotImplementedError('Implement AE evaluation loop')
            # TODO END
    numel = xin[0].numel()
    return {'bce': tb/n, 'mse': tm/(n*numel), 'mae': ta/(n*numel)}


Configure and train the one-digit Dense AE experiment.

Expected outcome: rapid decrease in reconstruction loss and clear reconstructions.


In [ ]:
# TODO START
dense_one = None
opt_dense_one = None
EPOCHS_ONE = 20
# TODO END
hist_dense_one = fit_or_load_model(
    model=dense_one,
    run_name='mnist_one_dense_ae',
    train_fn=train_autoencoder,
    load_if_available=USE_SAVED_MODELS_IF_AVAILABLE,
    loader=train_one_loader,
    optimizer=opt_dense_one,
    epochs=EPOCHS_ONE,
    flatten_input=True,
)
metrics_dense_one = evaluate_autoencoder(dense_one, test_one_loader, flatten_input=True)


Summarize one-digit Dense AE results.

Read the final metrics, then check the loss curve and reconstruction quality.


In [ ]:
def show_reconstructions(model, loader, flatten_input, title='Recon', n_show=8):
    model.eval()
    x, _ = next(iter(loader))
    x = x.to(device)
    xin = x.view(x.size(0), -1) if flatten_input else x
    with torch.no_grad():
        xhat = model(xin)
    if flatten_input:
        xhat = xhat.view(-1, 1, 28, 28)

    x = x[:n_show].cpu().numpy()
    xhat = xhat[:n_show].cpu().numpy()

    fig, axes = plt.subplots(2, n_show, figsize=(1.5*n_show, 3.2))
    for i in range(n_show):
        axes[0, i].imshow(x[i, 0], cmap='gray')
        axes[1, i].imshow(xhat[i, 0], cmap='gray')
        axes[0, i].axis('off')
        axes[1, i].axis('off')
    axes[0, 0].set_ylabel('real')
    axes[1, 0].set_ylabel('recon')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_reconstructions_vae(model, loader, title='VAE recon', n_show=8):
    model.eval()
    x, _ = next(iter(loader))
    x = x.to(device)

    with torch.no_grad():
        xhat, _, _ = model(x)

    if x.dim() == 2:
        x = x.view(-1, 1, 28, 28)
    if xhat.dim() == 2:
        xhat = xhat.view(-1, 1, 28, 28)

    x = x[:n_show].cpu().numpy()
    xhat = xhat[:n_show].cpu().numpy()

    fig, axes = plt.subplots(2, n_show, figsize=(1.5*n_show, 3.2))
    for i in range(n_show):
        axes[0, i].imshow(x[i, 0], cmap='gray')
        axes[1, i].imshow(xhat[i, 0], cmap='gray')
        axes[0, i].axis('off')
        axes[1, i].axis('off')
    axes[0, 0].set_ylabel('real')
    axes[1, 0].set_ylabel('recon')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_epoch_metric(histories, metric_key, title, ylabel):
    plt.figure(figsize=(5.0, 3.5))
    for name, hist in histories.items():
        plt.plot(range(1, len(hist)+1), [h[metric_key] for h in hist], marker='o', label=name)
    plt.xlabel('Epoch')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def report_autoencoder_metrics(name, history, metrics):
    print(f"[{name}] final train recon BCE: {history[-1]['train_recon_bce']:.4f}")
    print(f"[{name}] test metrics: {metrics}")


def report_vae_metrics(name, history, metrics):
    h = history[-1]
    print(f"[{name}] final train total: {h['train_loss']:.4f}")
    print(f"[{name}] final train recon: {h['train_recon_bce']:.4f}")
    print(f"[{name}] final train KL: {h['train_kl']:.4f}")
    print(f"[{name}] test metrics: {metrics}")


In [ ]:
report_autoencoder_metrics('One-digit Dense AE', hist_dense_one, metrics_dense_one)
plot_epoch_metric({'One-digit Dense AE': hist_dense_one}, 'train_recon_bce', 'One-digit Dense AE', 'Train recon BCE')
show_reconstructions(dense_one, test_one_loader, flatten_input=True, title='One-digit Dense AE')


## 2 - MNIST all digits: Dense AE vs Convolutional AE

Instantiate all-digits MNIST loaders.

This increases difficulty and tests model capacity across multiple digit styles.


### Convolutional AE pseudocode (this section)

```text
input: x with shape [B, 1, 28, 28]
z = ConvEncoder(x)
x_hat = ConvDecoder(z)
L_recon = BCE(x_hat, x)
update parameters with backprop
```

Compared to dense AE, the convolutional model keeps spatial locality explicitly.


In [ ]:
train_all_loader, test_all_loader = build_digit_loaders(
    dataset_name='mnist',
    target_class=None,
    batch_size=128,
    train_limit=20000,
    test_limit=4000,
)
plot_batch_examples(train_all_loader, title='MNIST all digits')


Configure and train Dense AE on all MNIST digits.

Keep this as the dense baseline for later comparison.


In [ ]:
# TODO START
dense_all = None
opt_dense_all = None
EPOCHS_ALL_DENSE = 20
# TODO END
hist_dense_all = fit_or_load_model(
    model=dense_all,
    run_name='mnist_all_dense_ae',
    train_fn=train_autoencoder,
    load_if_available=USE_SAVED_MODELS_IF_AVAILABLE,
    loader=train_all_loader,
    optimizer=opt_dense_all,
    epochs=EPOCHS_ALL_DENSE,
    flatten_input=True,
)
metrics_dense_all = evaluate_autoencoder(dense_all, test_all_loader, flatten_input=True)


### Convolutional AE construction recipe 

Build an encoder-decoder with these components:

Encoder:
1. `Conv2d(1 -> 32, kernel=3, stride=2, padding=1)` + `BatchNorm2d(32)` + `ReLU`
2. `Conv2d(32 -> 64, kernel=3, stride=2, padding=1)` + `BatchNorm2d(64)` + `ReLU`
3. `Conv2d(64 -> 128, kernel=3, stride=1, padding=1)` + `BatchNorm2d(128)` + `ReLU`
4. Flatten (`128 x 7 x 7`) and project with `Linear(128*7*7 -> latent_dim)`

Decoder:
1. `Linear(latent_dim -> 128*7*7)` and reshape to `128 x 7 x 7`
2. `ConvTranspose2d(128 -> 64, kernel=3, stride=1, padding=1)` + `ReLU`
3. `ConvTranspose2d(64 -> 32, kernel=3, stride=2, padding=1, output_padding=1)` + `ReLU`
4. `ConvTranspose2d(32 -> 1, kernel=3, stride=2, padding=1, output_padding=1)` + `Sigmoid`

Use this as the convolutional model for all comparisons.

Define the Convolutional Autoencoder used in comparisons.

Focus on spatial feature extraction (Conv2d) and image reconstruction (ConvTranspose2d).


In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        # TODO START
        raise NotImplementedError('Implement ConvAutoencoder layers from the recipe')
        # TODO END

    def encode(self, x):
        h = self.enc_conv(x)
        return self.enc_fc(h.view(h.size(0), -1))

    def decode(self, z):
        h = self.dec_fc(z).view(-1, 128, 7, 7)
        return self.dec_conv(h)

    def forward(self, x):
        return self.decode(self.encode(x))


Configure and train the Convolutional AE on all MNIST digits.

Use the same data split and epoch budget as Dense AE for fair comparison.


In [ ]:
# TODO START
conv_all = None
opt_conv_all = None
EPOCHS_ALL_CONV = 20
# TODO END
hist_conv_all = fit_or_load_model(
    model=conv_all,
    run_name='mnist_all_conv_ae',
    train_fn=train_autoencoder,
    load_if_available=USE_SAVED_MODELS_IF_AVAILABLE,
    loader=train_all_loader,
    optimizer=opt_conv_all,
    epochs=EPOCHS_ALL_CONV,
    flatten_input=False,
)
metrics_conv_all = evaluate_autoencoder(conv_all, test_all_loader, flatten_input=False)


Compare Dense AE and Convolutional AE on all MNIST digits.

Inspect both metric trends and visual reconstructions before concluding.


In [ ]:
report_autoencoder_metrics('MNIST Dense AE', hist_dense_all, metrics_dense_all)
report_autoencoder_metrics('MNIST Convolutional AE', hist_conv_all, metrics_conv_all)

plot_epoch_metric(
    {'Dense AE': hist_dense_all, 'Convolutional AE': hist_conv_all},
    'train_recon_bce',
    'MNIST: Dense vs Convolutional AE',
    'Train recon BCE'
)

show_reconstructions(dense_all, test_all_loader, flatten_input=True, title='MNIST Dense AE')
show_reconstructions(conv_all, test_all_loader, flatten_input=False, title='MNIST Convolutional AE')


### MNIST latent traversal setup (Dense + Convolutional)


### Latent traversal pseudocode (this section)

```text
choose latent sampling range [low, high]
sample n_samples points in 2D latent space
for each latent point z:
    x_hat = decoder(z)
visualize all generated images in a panel
```

For non-2D latent models, we wrap the decoder by filling extra latent dimensions with zeros.


In [ ]:
# TODO START
mnist_dense_decoder = None
mnist_conv_decoder = None
# TODO END


Render MNIST latent traversals.

Observe continuity and structure differences between Dense and Convolutional decoders.


In [ ]:
def plot_latent_traversal(decoder_fn, title='Latent traversal', n_samples=200, low=-2.5, high=2.5):
    grid = int(np.sqrt(n_samples))
    n_rows, n_cols = grid, grid
    z1 = torch.linspace(low, high, n_cols)
    z2 = torch.linspace(low, high, n_rows)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.0*n_cols, 1.0*n_rows))
    with torch.no_grad():
        for i, yy in enumerate(z2):
            for j, xx in enumerate(z1):
                z = torch.tensor([[xx, yy]], dtype=torch.float32, device=device)
                xhat = decoder_fn(z)
                if xhat.dim() == 2:
                    xhat = xhat.view(-1, 1, 28, 28)
                img = xhat[0, 0].detach().cpu().numpy()
                axes[i, j].imshow(img, cmap='gray', vmin=0, vmax=1)
                axes[i, j].axis('off')

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_latent_traversal(mnist_dense_decoder, title='MNIST Dense AE traversal', n_samples=200)
plot_latent_traversal(mnist_conv_decoder, title='MNIST Convolutional AE traversal', n_samples=200)


## 3 - Convolutional VAE on all MNIST digits

Design choice: reuse the convolutional encoder/decoder idea from the Convolutional AE,
and add probabilistic latent heads (`mu`, `logvar`) plus reparameterization.

Define the convolutional VAE architecture.

Pay attention to the two latent heads (`mu`, `logvar`) and reparameterization path.


### VAE formulation and algorithm (this section)

Latent posterior approximation:

$$
q_\phi(z\mid x)=\mathcal{N}\left(z;\mu_\phi(x),\operatorname{diag}(\sigma_\phi^2(x))\right)
$$

Reparameterization trick:

$$
\epsilon\sim\mathcal{N}(0,I),\quad z=\mu_\phi(x)+\sigma_\phi(x)\odot\epsilon
$$

Decoder likelihood model (Bernoulli-style reconstruction for normalized pixels):

$$
p_\theta(x\mid z)
$$

Training objective used in the notebook:

$$
\mathcal{L}_{\text{total}}=\mathcal{L}_{\text{recon}}+\beta\,\mathcal{L}_{\text{KL}}
$$

with

$$
\mathcal{L}_{\text{recon}}=\operatorname{BCE}(\hat{x},x),\qquad
\mathcal{L}_{\text{KL}}=-\frac{1}{2}\sum_j\left(1+\log\sigma_j^2-\mu_j^2-\sigma_j^2\right)
$$


How $\beta$ changes VAE behavior:

- $\beta < 1$: prioritizes reconstruction, usually sharper outputs, weaker latent regularization.
- $\beta \approx 1$: standard ELBO-style balance between reconstruction and latent structure.
- $\beta > 1$: stronger regularization toward $\mathcal{N}(0, I)$, often smoother traversal/sampling but blurrier reconstructions.

In practice, increase $\beta$ if latent space looks disorganized; decrease $\beta$ if reconstructions are too weak.

Pseudocode:

```text
h = ConvEncoderBackbone(x)
mu = Head_mu(h)
logvar = Head_logvar(h)
std = exp(0.5 * logvar)
eps ~ N(0, I)
z = mu + std * eps
x_hat = ConvDecoder(z)
loss = BCE(x_hat, x) + beta * KL(mu, logvar)
update parameters
```


In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim

        self.enc_conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )

        # TODO START
        # Add mean and log-variance heads from flattened conv features.
        raise NotImplementedError('Implement ConvVAE latent heads')
        # TODO END

        self.dec_fc = nn.Linear(latent_dim, 128 * 7 * 7)
        self.dec_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=1, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.enc_conv(x)
        h = h.view(h.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        # TODO START
        raise NotImplementedError('Implement reparameterization')
        # TODO END

    def decode(self, z):
        h = self.dec_fc(z).view(-1, 128, 7, 7)
        return self.dec_conv(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        xhat = self.decode(z)
        return xhat, mu, logvar


Define VAE objective and training/evaluation loops.

This introduces KL regularization in addition to reconstruction loss.


In [ ]:
def vae_loss(xhat, x, mu, logvar, beta=0.7):
    # TODO START
    raise NotImplementedError('Implement VAE loss')
    # TODO END


def train_vae(model, loader, optimizer, epochs=20, beta=0.7):
    model.train()
    hist = []
    for ep in range(epochs):
        tl, tr, tk = 0.0, 0.0, 0.0
        for x, _ in tqdm(loader, leave=False):
            x = x.to(device)
            # TODO START
            raise NotImplementedError('Implement VAE training step')
            # TODO END
            tl += loss.item() * x.size(0)
            tr += recon.item() * x.size(0)
            tk += kl.item() * x.size(0)
        n = len(loader.dataset)
        hist.append({'train_loss': tl/n, 'train_recon_bce': tr/n, 'train_kl': tk/n})
        print(f'Epoch {ep+1}/{epochs} | train_loss={tl/n:.4f} train_recon={tr/n:.4f} train_kl={tk/n:.4f}')
    return hist


def evaluate_vae(model, loader, beta=0.7):
    model.eval()
    tl, tr, tk, tm, ta, n = 0.0, 0.0, 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            xhat, mu, logvar = model(x)
            b = x.size(0)
            loss, recon, kl = vae_loss(xhat, x, mu, logvar, beta=beta)
            # TODO START
            raise NotImplementedError('Implement VAE evaluation loop')
            # TODO END
    numel = x[0].numel()
    return {'loss': tl/n, 'recon_bce': tr/n, 'kl': tk/n, 'mse': tm/(n*numel), 'mae': ta/(n*numel)}


Configure and train the convolutional VAE on all MNIST digits.

Use this to compare against Dense AE and Convolutional AE under the same epoch budget.


In [ ]:
# TODO START
vae_all = None
opt_vae_all = None
EPOCHS_VAE_ALL = 20
BETA_VAE_ALL = 0.7
# TODO END
hist_vae_all = fit_or_load_model(
    model=vae_all,
    run_name='mnist_all_conv_vae',
    train_fn=train_vae,
    load_if_available=USE_SAVED_MODELS_IF_AVAILABLE,
    loader=train_all_loader,
    optimizer=opt_vae_all,
    epochs=EPOCHS_VAE_ALL,
    beta=BETA_VAE_ALL,
)
metrics_vae_all = evaluate_vae(vae_all, test_all_loader, beta=BETA_VAE_ALL)


Summarize MNIST 3-model comparison.

Read VAE loss decomposition (total/recon/KL), then compare against AE baselines.


In [ ]:
report_vae_metrics('MNIST VAE', hist_vae_all, metrics_vae_all)
plot_epoch_metric(
    {'Dense AE': hist_dense_all, 'Convolutional AE': hist_conv_all, 'VAE (recon)': hist_vae_all},
    'train_recon_bce',
    'MNIST: train reconstruction BCE (3 models)',
    'Train recon BCE'
)
plot_epoch_metric({'VAE': hist_vae_all}, 'train_kl', 'MNIST VAE train KL', 'Train KL')
show_reconstructions_vae(vae_all, test_all_loader, title='MNIST VAE')

print('=== MNIST comparison ===')
print('Dense AE:', metrics_dense_all)
print('Convolutional AE:', metrics_conv_all)
print('VAE:', metrics_vae_all)


### MNIST VAE traversal setup


In [ ]:
# TODO START
mnist_vae_decoder = None
# TODO END


Render MNIST latent traversals.

Observe continuity and structure differences between Dense and Convolutional decoders.


In [ ]:
plot_latent_traversal(mnist_dense_decoder, title='MNIST Dense AE traversal', n_samples=200)
plot_latent_traversal(mnist_conv_decoder, title='MNIST Convolutional AE traversal', n_samples=200)
plot_latent_traversal(mnist_vae_decoder, title='MNIST VAE traversal', n_samples=200)


## 4 - Fashion-MNIST: Convolutional AE vs VAE

Instantiate Fashion-MNIST loaders for the final comparison stage.

This tests transfer of model choices to a harder and more diverse dataset.


In [ ]:
train_fashion_loader, test_fashion_loader = build_digit_loaders(
    dataset_name='fashion',
    target_class=None,
    batch_size=128,
    train_limit=20000,
    test_limit=4000,
)
plot_batch_examples(train_fashion_loader, title='Fashion-MNIST')


Train Convolutional AE and VAE on Fashion-MNIST.

Keep training settings aligned to make comparison meaningful.


In [ ]:
# TODO START
conv_fashion = None
opt_conv_fashion = None
EPOCHS_FASHION_CONV = 20

vae_fashion = None
opt_vae_fashion = None
EPOCHS_FASHION_VAE = 20
BETA_FASHION_VAE = 0.7
# TODO END

hist_conv_fashion = fit_or_load_model(
    model=conv_fashion,
    run_name='fashion_conv_ae',
    train_fn=train_autoencoder,
    load_if_available=USE_SAVED_MODELS_IF_AVAILABLE,
    loader=train_fashion_loader,
    optimizer=opt_conv_fashion,
    epochs=EPOCHS_FASHION_CONV,
    flatten_input=False,
)
metrics_conv_fashion = evaluate_autoencoder(conv_fashion, test_fashion_loader, flatten_input=False)

hist_vae_fashion = fit_or_load_model(
    model=vae_fashion,
    run_name='fashion_conv_vae',
    train_fn=train_vae,
    load_if_available=USE_SAVED_MODELS_IF_AVAILABLE,
    loader=train_fashion_loader,
    optimizer=opt_vae_fashion,
    epochs=EPOCHS_FASHION_VAE,
    beta=BETA_FASHION_VAE,
)
metrics_vae_fashion = evaluate_vae(vae_fashion, test_fashion_loader, beta=BETA_FASHION_VAE)


Analyze Fashion-MNIST training curves and reconstruction outputs.

Use both metrics and visuals before moving to traversal and FID.


In [ ]:
report_autoencoder_metrics('Fashion Convolutional AE', hist_conv_fashion, metrics_conv_fashion)
report_vae_metrics('Fashion VAE', hist_vae_fashion, metrics_vae_fashion)
plot_epoch_metric({'Convolutional AE': hist_conv_fashion, 'VAE (recon)': hist_vae_fashion}, 'train_recon_bce', 'Fashion: train recon BCE', 'Train recon BCE')
plot_epoch_metric({'VAE': hist_vae_fashion}, 'train_kl', 'Fashion VAE train KL', 'Train KL')
show_reconstructions(conv_fashion, test_fashion_loader, flatten_input=False, title='Fashion Convolutional AE')
show_reconstructions_vae(vae_fashion, test_fashion_loader, title='Fashion VAE')

print('=== Fashion comparison ===')
print('Convolutional AE:', metrics_conv_fashion)
print('VAE:', metrics_vae_fashion)


### Fashion latent transitions between real samples

No retraining here. We reuse trained `conv_fashion` and `vae_fashion`.
Instead of sampling a fixed latent grid, we choose two real Fashion-MNIST examples and interpolate between their latent codes.

Interpolation rule with $\alpha \in [0,1]$:

$$
z(\alpha)=(1-\alpha)z_A+\alpha z_B
$$

Then decode each $z(\alpha)$ to observe how one item transitions into another.


### Fashion transition note

The models are already trained. This section only encodes two real inputs and decodes the latent interpolation path.

For VAE, using encoder mean $\mu(x)$ gives a deterministic transition.
If you set `use_vae_mu=False`, interpolation uses sampled latent points and becomes noisier.


In [ ]:
FASHION_LABELS = {
    0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat',
    5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'
}


def pick_transition_pair(loader, start_label=0, end_label=9, max_scan_batches=80):
    xs, ys = [], []
    for bi, (x, y) in enumerate(loader):
        xs.append(x)
        ys.append(y)
        if bi + 1 >= max_scan_batches:
            break

    x_all = torch.cat(xs, dim=0)
    y_all = torch.cat(ys, dim=0)

    def first_index_for_label(lbl):
        idx = torch.where(y_all == lbl)[0]
        if idx.numel() == 0:
            raise ValueError(f'Label {lbl} not found in scanned batches.')
        return int(idx[0].item())

    idx_a = 0 if start_label is None else first_index_for_label(start_label)

    if end_label is None:
        if x_all.size(0) < 2:
            raise ValueError('Need at least 2 samples to build a transition pair.')
        idx_b = 1 if idx_a == 0 else 0
    else:
        idx_b = first_index_for_label(end_label)
        if idx_b == idx_a:
            idx_alt = torch.where(y_all == end_label)[0]
            if idx_alt.numel() > 1:
                idx_b = int(idx_alt[1].item())

    x_a = x_all[idx_a:idx_a + 1]
    x_b = x_all[idx_b:idx_b + 1]
    y_a = int(y_all[idx_a].item())
    y_b = int(y_all[idx_b].item())
    return x_a, y_a, x_b, y_b


def encode_for_transition(model, x, use_vae_mu=True):
    z_enc = model.encode(x)
    if isinstance(z_enc, tuple):
        mu, logvar = z_enc
        if use_vae_mu:
            return mu
        return model.reparameterize(mu, logvar)
    return z_enc


def plot_latent_transition_between_samples(
    model,
    loader,
    title='Latent transition',
    n_steps=25,
    start_label=0,
    end_label=9,
    use_vae_mu=True,
):
    model.eval()
    x_a, y_a, x_b, y_b = pick_transition_pair(loader, start_label=start_label, end_label=end_label)

    x_a_dev = x_a.to(device)
    x_b_dev = x_b.to(device)

    with torch.no_grad():
        z_a = encode_for_transition(model, x_a_dev, use_vae_mu=use_vae_mu)
        z_b = encode_for_transition(model, x_b_dev, use_vae_mu=use_vae_mu)

        alphas = torch.linspace(0.0, 1.0, n_steps, device=device).unsqueeze(1)
        z_path = (1.0 - alphas) * z_a + alphas * z_b
        x_path = model.decode(z_path).clamp(0, 1).cpu()

    fig, axes = plt.subplots(2, n_steps, figsize=(1.15 * n_steps, 3.1))

    for j in range(n_steps):
        axes[0, j].axis('off')
        axes[1, j].imshow(x_path[j, 0], cmap='gray')
        axes[1, j].axis('off')
        if j in (0, n_steps - 1):
            axes[1, j].set_title(f'alpha={j/(n_steps-1):.2f}', fontsize=8)

    axes[0, 0].imshow(x_a[0, 0], cmap='gray')
    axes[0, 0].set_title(f'start: {FASHION_LABELS.get(y_a, y_a)}', fontsize=9)
    axes[0, -1].imshow(x_b[0, 0], cmap='gray')
    axes[0, -1].set_title(f'end: {FASHION_LABELS.get(y_b, y_b)}', fontsize=9)

    axes[0, 0].set_ylabel('real anchors')
    axes[1, 0].set_ylabel('decoded path')

    mode = 'mu path' if use_vae_mu else 'sampled path'
    plt.suptitle(f"{title} | {FASHION_LABELS.get(y_a, y_a)} -> {FASHION_LABELS.get(y_b, y_b)} | steps={n_steps} | {mode}")
    plt.tight_layout()
    plt.show()


Visualize Fashion latent transitions for Convolutional AE and VAE.

Configure `start_label`, `end_label`, and `n_steps` to control which categories are connected and how smooth the transition appears.


In [ ]:
FASHION_START_LABEL = 0
FASHION_END_LABEL = 9
FASHION_TRANSITION_STEPS = 25

plot_latent_transition_between_samples(
    conv_fashion,
    test_fashion_loader,
    title='Fashion Convolutional AE transition',
    n_steps=FASHION_TRANSITION_STEPS,
    start_label=FASHION_START_LABEL,
    end_label=FASHION_END_LABEL,
)

plot_latent_transition_between_samples(
    vae_fashion,
    test_fashion_loader,
    title='Fashion VAE transition',
    n_steps=FASHION_TRANSITION_STEPS,
    start_label=FASHION_START_LABEL,
    end_label=FASHION_END_LABEL,
    use_vae_mu=True,
)


## Final block) Fashion-MNIST FID (library)

Uses `torch_fidelity.calculate_metrics` (library-based FID).
FID uses CUDA when available; otherwise CPU (including MPS-only environments).


### FID evaluation explained

What this evaluation does:

1. Build a **real reference set** from Fashion-MNIST test images (`N_FID_SAMPLES`).
2. Build two **generated sets** with the same size:
 - samples decoded from Convolutional AE latent vectors
 - samples decoded from VAE prior latent vectors
3. Convert all images to the format required by the FID metric (`uint8`, 3 channels).
4. Use Inception features through `torch-fidelity` to estimate FID.
5. Compare feature distributions of:
 - real vs conv-generated
 - real vs vae-generated

Interpretation:

- FID measures distance between two feature distributions (real and generated).
- **Lower FID is better** (generated distribution closer to real distribution).
- FID is sensitive to sample count and preprocessing. Keep both identical across models for fair comparison.

Important caveat:

- FID is one quantitative signal, not a full quality guarantee.
- Always inspect reconstruction and traversal visuals together with FID.

Define helper functions for FID preparation and sample generation.

These functions format images and generate synthetic batches for evaluation.


In [ ]:
def to_uint8_3ch(x):
    x = x.clamp(0, 1)
    x = (x * 255.0).to(torch.uint8)
    return x.repeat(1, 3, 1, 1)


def collect_real_images(loader, n_samples=500):
    xs, count = [], 0
    for x, _ in loader:
        xs.append(x)
        count += x.size(0)
        if count >= n_samples:
            break
    return torch.cat(xs, dim=0)[:n_samples]


def sample_from_conv_empirical_latent(model, loader, n_samples=500):
    model.eval()
    zs, count = [], 0
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            z = model.encode(x)
            zs.append(z.cpu())
            count += z.size(0)
            if count >= max(n_samples, 2000):
                break
    z_all = torch.cat(zs, dim=0)
    z = z_all[torch.randperm(z_all.size(0))[:n_samples]].to(device)
    with torch.no_grad():
        xhat = model.decode(z).cpu().clamp(0, 1)
    return xhat


def sample_from_vae_prior(model, n_samples=500):
    model.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, model.latent_dim, device=device)
        xhat = model.decode(z).cpu().clamp(0, 1)
    return xhat


Compute library-based FID on Fashion-MNIST.

This block compares generated sample quality quantitatively using `torch-fidelity` FID.


In [ ]:
N_FID_SAMPLES = 200  # configurable

fid_use_cuda = torch.cuda.is_available()

real_imgs = collect_real_images(test_fashion_loader, n_samples=N_FID_SAMPLES)
conv_gen = sample_from_conv_empirical_latent(conv_fashion, train_fashion_loader, n_samples=N_FID_SAMPLES)
vae_gen = sample_from_vae_prior(vae_fashion, n_samples=N_FID_SAMPLES)

real_u8 = to_uint8_3ch(real_imgs)
conv_u8 = to_uint8_3ch(conv_gen)
vae_u8 = to_uint8_3ch(vae_gen)

import importlib
import subprocess
import sys
import tempfile
from pathlib import Path
from PIL import Image


def save_uint8_batch_to_dir(batch_u8, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for i in range(batch_u8.size(0)):
        arr = batch_u8[i].permute(1, 2, 0).cpu().numpy()
        Image.fromarray(arr).save(out_dir / f'{i:05d}.png')


def compute_fid_with_torch_fidelity(real_batch_u8, gen_batch_u8, use_cuda=False):
    try:
        import torch_fidelity
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fidelity>=0.3,<1.0'])
        importlib.invalidate_caches()
        import torch_fidelity

    with tempfile.TemporaryDirectory() as tmpdir:
        real_dir = Path(tmpdir) / 'real'
        gen_dir = Path(tmpdir) / 'gen'
        save_uint8_batch_to_dir(real_batch_u8, real_dir)
        save_uint8_batch_to_dir(gen_batch_u8, gen_dir)

        metrics = torch_fidelity.calculate_metrics(
            input1=str(real_dir),
            input2=str(gen_dir),
            cuda=bool(use_cuda),
            fid=True,
            isc=False,
            kid=False,
            verbose=False,
        )
    return float(metrics['frechet_inception_distance'])


fid_conv = compute_fid_with_torch_fidelity(real_u8, conv_u8, use_cuda=fid_use_cuda)
fid_vae = compute_fid_with_torch_fidelity(real_u8, vae_u8, use_cuda=fid_use_cuda)

print(f'FID backend: torch-fidelity')
print(f'FID device: {"cuda" if fid_use_cuda else "cpu"}')
print(f'FID Convolutional AE (Fashion, n={N_FID_SAMPLES}): {fid_conv:.4f}')
print(f'FID VAE             (Fashion, n={N_FID_SAMPLES}): {fid_vae:.4f}')


## Final reflection

1. How did Dense vs Convolutional AE compare on MNIST?
2. Did the convolutional VAE close the gap with AE models?
3. What does the KL curve suggest about latent regularization?
4. In Fashion-MNIST, which model had better FID and does it match visual quality?


Answers:
